# 11 · Mixed precision, mixup, and knowledge distillation

**Prerequisites:** training loops, cross-entropy, and controlled experiments.
**Time:** 40–60 minutes including exercises. Offline data; GPU is optional.

You will use modern `torch.amp`, mix images with correctly paired labels,
and teach a small student from a frozen teacher. We run controlled short
comparisons and inspect validation results. These techniques solve different
problems and are not automatic accuracy improvements.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from course_utils import get_digits_loaders, TinyCNN, fit, evaluate
epochs = 1 if SMOKE_TEST else 6
train_loader, val_loader, test_loader = get_digits_loaders(batch_size=64, seed=42)
use_amp = device.type == 'cuda'
print('CUDA float16 AMP enabled:', use_amp, '| CPU runs float32.')


## 1. Understand autocasting and loss scaling

Autocast selects lower precision for eligible operations; model parameters
remain float32. `GradScaler` scales float16 losses to reduce gradient underflow.
It unscales before optimizer updates and can skip updates with nonfinite
gradients. If clipping, unscale first. Our CPU fallback disables autocast
and scaling; it does not depend on CPU bfloat16 support.

AMP helps many larger CUDA workloads, but tiny 8×8 models may be dominated
by overhead. Measure after warm-up with CUDA synchronization; do not infer a
GPU speedup from a single noisy notebook run.


In [ ]:
demo_model = TinyCNN().to(device)
demo_optimizer = torch.optim.Adam(demo_model.parameters(), lr=0.003)
demo_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
demo_optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
    demo_logits = demo_model(images)
    demo_loss = F.cross_entropy(demo_logits, labels)
demo_scaler.scale(demo_loss).backward()
demo_scaler.unscale_(demo_optimizer)
gradient_norm = nn.utils.clip_grad_norm_(demo_model.parameters(), max_norm=5.0)
demo_scaler.step(demo_optimizer)
demo_scaler.update()
print('Parameter dtype:', next(demo_model.parameters()).dtype)
print('Logit dtype:', demo_logits.dtype, '| loss:', demo_loss.item(), '| gradient norm:', float(gradient_norm))


## 2. Mix two images and both targets

Draw λ from a Beta distribution and form `mixed_x = λ*x + (1-λ)*x[perm]`.
The supervised loss is `λ*CE(logits,y) + (1-λ)*CE(logits,y[perm])`.
Keeping only the first label would teach contradictory targets. We use a
single λ per batch for clarity; mixed-image accuracy is not ordinary accuracy.


In [ ]:
def mixup_batch(x, y, alpha=0.4):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    permutation = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1.0 - lam) * x[permutation], y, y[permutation], lam

mixed, y_a, y_b, lam = mixup_batch(images, labels)
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(mixed[i, 0].cpu(), cmap='gray')
    ax.set_title(f'{lam:.2f}×{y_a[i].item()} + {1-lam:.2f}×{y_b[i].item()}')
    ax.axis('off')
plt.tight_layout(); plt.show()


## 3. Train with a switch for AMP and mixup

Validation always uses clean images and ordinary labels. Save the lowest
validation-loss state so every configuration uses the same selection rule.
Synchronization brackets timing because CUDA work is asynchronous. These
short timings include validation and checkpoint copies and are not benchmarks.


In [ ]:
def train_mixup(alpha=0.0, amp=False):
    seed_everything(42)
    train, val, _ = get_digits_loaders(batch_size=64, seed=42)
    model = TinyCNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    enabled = amp and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=enabled)
    best_loss, best_state, history = float('inf'), None, []
    if device.type == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for x, y in train:
            x, y = x.to(device), y.to(device)
            x, first, second, mix = mixup_batch(x, y, alpha)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=enabled):
                logits = model(x)
                loss = mix * F.cross_entropy(logits, first) + (1-mix) * F.cross_entropy(logits, second)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        result = evaluate(model, val, device=device)
        history.append({'epoch': epoch + 1, 'val_loss': result['loss'], 'val_accuracy': result['accuracy']})
        if result['loss'] < best_loss:
            best_loss, best_state = result['loss'], copy.deepcopy(model.state_dict())
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    model.load_state_dict(best_state)
    return model, history, elapsed


In [ ]:
fp32_model, fp32_history, fp32_seconds = train_mixup(alpha=0.0, amp=False)
amp_model, amp_history, amp_seconds = train_mixup(alpha=0.0, amp=True)
mixup_model, mixup_history, mixup_seconds = train_mixup(alpha=0.4, amp=True)
cnn_models = {'CNN float32': fp32_model, 'CNN AMP': amp_model, 'CNN AMP + mixup': mixup_model}
cnn_table = []
for (name, model), seconds in zip(cnn_models.items(), [fp32_seconds, amp_seconds, mixup_seconds]):
    result = evaluate(model, val_loader, device=device)
    cnn_table.append({'name': name, 'val_loss': result['loss'],
                      'val_accuracy': result['accuracy'], 'seconds_including_validation': seconds})
display(pd.DataFrame(cnn_table))
print('On CPU, the first two runs use the same float32 path.')


## 4. Define a student and freeze a teacher

We reuse the trained float32 CNN as teacher. The student is a smaller MLP.
First train a hard-label-only student baseline; then train the same student
initialization with distillation. Teacher quality matters: an undertrained
teacher can transfer bad predictions, and the smoke-test teacher is intentionally
weak. Check parameter counts rather than assuming one model is smaller.


In [ ]:
teacher = fp32_model
teacher.eval()
teacher.requires_grad_(False)
teacher.zero_grad(set_to_none=True)

class Student(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
    def forward(self, x):
        return self.net(x)

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

print('Teacher parameters:', parameter_count(teacher))
print('Student parameters:', parameter_count(Student()))
print('Teacher validation accuracy:', evaluate(teacher, val_loader, device=device)['accuracy'])


## 5. Build the distillation loss

Hard labels use ordinary cross-entropy. Soft targets use temperature-softened
teacher probabilities and student log-probabilities. PyTorch KL divergence
expects log-probabilities as its input; `reduction='batchmean'` divides by
batch size. Multiply the soft loss by `T²` to compensate for temperature's
gradient scaling. Teacher logits are detached and never backpropagated.

`alpha` below is the soft-target contribution, so alpha=0 is the supervised
baseline. Keep temperature and alpha fixed before evaluating final test data.


In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, temperature=3.0, alpha=0.5):
    hard = F.cross_entropy(student_logits, labels)
    soft = F.kl_div(F.log_softmax(student_logits / temperature, dim=1),
                    F.softmax(teacher_logits.detach() / temperature, dim=1),
                    reduction='batchmean') * temperature ** 2
    return (1-alpha) * hard + alpha * soft, hard.detach(), soft.detach()

with torch.no_grad():
    teacher_logits = teacher(images)
demo_student = Student().to(device)
loss, hard, soft = distillation_loss(demo_student(images), teacher_logits, labels)
print({'combined': loss.item(), 'hard_ce': hard.item(), 'soft_kl_scaled': soft.item()})


In [ ]:
def train_student(alpha):
    seed_everything(42)
    train, val, _ = get_digits_loaders(batch_size=64, seed=42)
    student = Student().to(device)
    optimizer = torch.optim.Adam(student.parameters(), lr=0.003)
    best_loss, best_state = float('inf'), None
    for epoch in range(epochs):
        student.train()
        for x, y in train:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                teacher_output = teacher(x)
            optimizer.zero_grad(set_to_none=True)
            loss, _, _ = distillation_loss(student(x), teacher_output, y,
                                           temperature=3.0, alpha=alpha)
            loss.backward()
            optimizer.step()
        result = evaluate(student, val, device=device)
        if result['loss'] < best_loss:
            best_loss, best_state = result['loss'], copy.deepcopy(student.state_dict())
    student.load_state_dict(best_state)
    return student

hard_student = train_student(alpha=0.0)
distilled_student = train_student(alpha=0.5)
# Freeze the teacher's behavior and verify no teacher gradient was created.
assert all(p.grad is None for p in teacher.parameters())


## 6. Compare students fairly, then evaluate the selected student

Both students see identical splits, initialization, batch order, epochs,
and optimizer settings. Choose by validation cross-entropy, then perform
one final test evaluation. This is a small educational comparison, not a
general claim that distillation beats supervised learning. It also required
a teacher's training and inference cost, which should be reported in practice.


In [ ]:
students = {'hard labels': hard_student, 'distilled': distilled_student}
student_rows = []
for name, model in students.items():
    result = evaluate(model, val_loader, device=device)
    student_rows.append({'student': name, 'val_loss': result['loss'],
                         'val_accuracy': result['accuracy'], 'parameters': parameter_count(model)})
student_table = pd.DataFrame(student_rows)
display(student_table)
chosen_name = student_table.loc[student_table['val_loss'].idxmin(), 'student']
final = evaluate(students[chosen_name], test_loader, device=device)
print('Selected:', chosen_name, '| test loss:', final['loss'], '| test accuracy:', final['accuracy'])
out_dir = ARTIFACTS / '11_training_techniques'; out_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(cnn_table).to_csv(out_dir / 'cnn_validation.csv', index=False)
student_table.to_csv(out_dir / 'student_validation.csv', index=False)


## Practice 1 · Verify the mixup endpoints

Show that λ=1 equals the first-label loss and λ=0 equals the second-label
loss. Change the labels and observe that the identities still hold. This
is an algebra check, not a training run.


In [ ]:
toy_logits = torch.tensor([[2.0, -1.0], [-0.5, 1.5]])
toy_a, toy_b = torch.tensor([0, 1]), torch.tensor([1, 0])
ce_a, ce_b = F.cross_entropy(toy_logits, toy_a), F.cross_entropy(toy_logits, toy_b)
for coefficient in [0.0, 0.5, 1.0]:
    print(coefficient, (coefficient * ce_a + (1-coefficient) * ce_b).item())
assert torch.allclose(1.0 * ce_a + 0.0 * ce_b, ce_a)
assert torch.allclose(0.0 * ce_a + 1.0 * ce_b, ce_b)


<details><summary>Solution idea</summary>
The mixed loss interpolates between two valid cross-entropies. Feeding a
single integer label for every mixed image loses the second target.
</details>

## Practice 2 · See what temperature does

Plot teacher probabilities for one validation image at different temperatures.
Higher temperatures soften the distribution. Argmax remains unchanged for
any positive temperature; soft relative preferences become more visible.


In [ ]:
sample_x, sample_y = next(iter(val_loader))
with torch.no_grad():
    sample_logits = teacher(sample_x[:1].to(device)).float().cpu()[0]
for t in [1.0, 3.0, 8.0]:
    plt.plot(range(10), (sample_logits / t).softmax(0), 'o-', label=f'T={t}')
plt.xticks(range(10)); plt.xlabel('Digit'); plt.ylabel('Probability')
plt.title(f'Teacher probabilities; true digit {sample_y[0].item()}')
plt.legend(); plt.show()


<details><summary>Solution idea</summary>
As temperature increases, softmax approaches a uniform distribution. Very
high temperature can dilute useful information. Distillation temperature
and post-training calibration temperature serve different objectives.
</details>

## Practice 3 · Calculate memory and plan a real AMP benchmark

Compute activation storage for a hypothetical batch at float32 and float16.
Then use a larger real workload to benchmark after warm-up. Arithmetic below
estimates only the named tensor; parameters, gradients, optimizer state,
saved activations, and allocator overhead add memory.


In [ ]:
batch, channels, height, width = 32, 64, 56, 56
elements = batch * channels * height * width
for dtype in [torch.float32, torch.float16]:
    megabytes = elements * torch.tensor([], dtype=dtype).element_size() / 1024**2
    print(dtype, f'{megabytes:.1f} MiB for this activation tensor')
if device.type == 'cuda':
    print('Current allocated GPU memory (MiB):', torch.cuda.memory_allocated() / 1024**2)


<details><summary>Solution idea</summary>
Float16 uses half the storage per element. AMP does not halve all training
memory because float32 weights, optimizer states, and some operations remain.
Benchmark the same model and batch shape with warm-up, synchronized timings,
multiple repetitions, and peak memory reset immediately before measurement.
</details>

**Completion check:** describe autocast versus scaling, implement paired-label
mixup, identify KL argument order, and explain detached teacher logits and T².

**Primary sources:** [torch.amp](https://docs.pytorch.org/docs/2.11/amp.html),
[mixup paper](https://arxiv.org/abs/1710.09412),
[PyTorch distillation tutorial](https://docs.pytorch.org/tutorials/beginner/knowledge_distillation_tutorial.html),
[Hinton et al., distillation](https://arxiv.org/abs/1503.02531).

**Next:** [12 · Build a small vision transformer](12_vision_transformer.ipynb).
